In [1]:
# ==============================================================================
# ПРАКТИКА 8 (Варіант 6: Чернігів)
# ==============================================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# ПІДГОТОВКА ДАНИХ (Tidy Data)
# ------------------------------------------------------------------------------
np.random.seed(42)
base_temp = 8.0     # Середньорічна температура (Варіант 6)
amplitude = 13.0    # Сезонна амплітуда (Варіант 6)
city = "Чернігів"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)
print("=== Перші 5 рядків набору climate ===")
print(climate.head())
print("\n" + "="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. groupby і agg за роками
# ------------------------------------------------------------------------------
yearly_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
print("=== Завдання 1: Статистика за роками ===")
print(yearly_stats)

print("\n[Висновок Завдання 1]:")
print("Середньорічні значення коливаються в межах ~7.6°C - 8.2°C без явного монотонного зростання")
print("чи падіння. Отже, вираженого тренду потепління немає, коливання є випадковими.\n")
print("="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. groupby і agg за місяцями
# ------------------------------------------------------------------------------
monthly_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
print("=== Завдання 2: Статистика за місяцями ===")
print(monthly_stats)

max_std_month = monthly_stats["std"].idxmax()
print(f"\n[Висновок Завдання 2]:")
print(f"Найбільший розкид (std = {monthly_stats['std'].max():.2f}) спостерігається у місяці №{max_std_month}.")
print("Перехідні сезони (весна/осінь) мають найбільшу нестабільність через активні циркуляційні зміни.\n")
print("="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. pivot_table
# ------------------------------------------------------------------------------
pivot_climate = climate.pivot_table(index="місяць", columns="рік", values="температура", aggfunc="mean")
print("=== Завдання 3: Зведена таблиця (pivot_table) ===")
print(pivot_climate)

print("\n[Висновок Завдання 3]:")
print("Зведена форма зручніша для сприйняття людиною (компактна матриця).")
print("Довга форма (tidy) зручніша для автоматизованої обробки, фільтрації та графіків у Seaborn.\n")
print("="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Похідні категорії і crosstab
# ------------------------------------------------------------------------------
def get_season(month):
    if month in [12, 1, 2]: return "зима"
    elif month in [3, 4, 5]: return "весна"
    elif month in [6, 7, 8]: return "літо"
    else: return "осінь"

climate["сезон"] = climate["місяць"].apply(get_season)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

ct = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
print("=== Завдання 4: Таблиця сопряженности (crosstab) ===")
print(ct)

print("\n[Висновок Завдання 4]:")
print("Розподіл відповідає очікуванням: усі 12 літніх місяців є 'теплішими за середнє' (True),")
print("а всі 12 зимових — 'холоднішими за середнє' (False).\n")
print("="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. pivot() на своєму наборі
# ------------------------------------------------------------------------------
pivoted_direct = climate.pivot(index="місяць", columns="рік", values="температура")
print("=== Завдання 5: Прямий pivot() ===")
print(pivoted_direct.head())

print("\n[Висновок Завдання 5]:")
print("pivot() працює без помилок, оскільки кожна комбінація (місяць, рік) є унікальною.")
print("Якби ми додали другу метеостанцію, з'явилися б дублікати параметрів, і pivot() падав би з ValueError.")
print("\n" + "="*80 + "\n")


# ------------------------------------------------------------------------------
# ВІДПОВІДІ НА КОНТРОЛЬНІ ПИТАННЯ
# ------------------------------------------------------------------------------
print("""=== КОНТРОЛЬНІ ПИТАННЯ ===

1. Логіка Split-Apply-Combine в groupby():
   - Split: дані розбиваються на окремі групи за значеннями вибраного ключа.
   - Apply: до кожної групи окремо застосовується обчислювальна функція (mean, std тощо).
   - Combine: отримані окремі результати збираються в єдину підсумкову структуру pandas.

2. Відмінність між pivot() і pivot_table():
   - pivot() вимагає строго унікальних пар (індекс, стовпець). При наявності дублікатів викликає помилку.
   - pivot_table() автоматично агрегує дублюючі записи за допомогою функції aggfunc (за замовчуванням mean).

3. Що показує crosstab() і чим відрізняється від groupby().size():
   - crosstab() будує зручну двовимірну матрицю частот перетину двох категоріальних змінних.
   - groupby().size() повертає аналогічні дані у вигляді одного довгого Series з мультиіндексом.

4. Чому потрібен довгий (tidy) формат:
   - Довгий формат (1 зміна = 1 стовпець) є універсальним стандартом для аналізу даних. 
     Він дозволяє легко виконувати будь-які групування, фільтрації та побудови графіків.
""")

=== Перші 5 рядків набору climate ===
      місто   рік  місяць  температура
0  Чернігів  2021       1         -4.5
1  Чернігів  2021       2         -3.4
2  Чернігів  2021       3          2.1
3  Чернігів  2021       4          9.5
4  Чернігів  2021       5         14.3


=== Завдання 1: Статистика за роками ===
          mean  min   max
рік                      
2021  8.283333 -4.5  22.6
2022  7.408333 -5.2  20.1
2023  7.808333 -5.5  21.1
2024  7.666667 -5.2  20.9

[Висновок Завдання 1]:
Середньорічні значення коливаються в межах ~7.6°C - 8.2°C без явного монотонного зростання
чи падіння. Отже, вираженого тренду потепління немає, коливання є випадковими.


=== Завдання 2: Статистика за місяцями ===
          mean       std
місяць                  
1       -4.900  0.424264
2       -4.225  1.132475
3        0.600  1.023067
4        8.375  0.865544
5       14.225  0.727438
6       19.250  0.300000
7       21.000  1.116542
8       19.475  1.408013
9       14.375  1.250000
10       7.625 